In [45]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [4]:
data = pd.read_csv('weatherAUS.csv')
data.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [5]:
missing_frac = data.isna().mean()

cols_to_drop = missing_frac[missing_frac > 0.40].index

data = data.drop(columns=cols_to_drop)

In [6]:
data[['RainToday', 'RainTomorrow']] = data[['RainToday', 'RainTomorrow']].replace({
    'Yes': 1,
    'No': 0
})

/var/folders/6k/1096gq_54_dgk0_cjz4762g40000gn/T/ipykernel_71184/1731782630.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[['RainToday','RainTomorrow']] = data[['RainToday','RainTomorrow']].replace({


In [7]:
print("Среднее арифметическое:", round(data['RainToday'].mean(), 2))

Среднее арифметическое: 0.22


In [8]:
data['Date'] = pd.to_datetime(data['Date'])
data['Month'] = data['Date'].dt.month
data = data.drop('Date', axis=1)

In [13]:
rain_frac = (
    data
    .groupby('Month')['RainToday']
    .agg(lambda x: x.sum() / x.count())
)

best_month = rain_frac.idxmax()
best_ratio = rain_frac.max()

print(f"Месяц с наибольшей долей дождливых дней: {best_month}")
print(f"Доля дождливых дней в этом месяце: {best_ratio:.2%}")

Месяц с наибольшей долей дождливых дней: 7
Доля дождливых дней в этом месяце: 27.07%


In [15]:
categoricals = ['Month', 'Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']
data = pd.get_dummies(data, columns=categoricals)
data.shape

(145460, 124)

In [16]:
clean_data = data.dropna()

In [17]:
X = clean_data.drop(columns=['RainTomorrow'])
y = clean_data['RainTomorrow']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=31)

In [20]:
print("Среднее значение целевой переменной:", round(y_test.mean(), 2))

Среднее значение целевой переменной: 0.23


In [26]:
np.random.seed(31)
random_idx = []
mean_min_temps = []

for _ in range(1000):
    idx = np.random.randint(0, X_train.shape[0], X_train.shape[0])
    random_idx.append(idx)

for idx in random_idx:
    mean_min_temp = data.loc[idx, 'MinTemp'].mean()
    mean_min_temps.append(mean_min_temp)

In [28]:
print('стандартного отклонения:', round(np.std(mean_min_temps), 2))

стандартного отклонения: 0.03


In [34]:
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [35]:
print("Roc auc:", round(roc_auc_score(y_test, y_pred), 2))

Roc auc: 0.73


In [43]:
params = {
    'max_leaf_nodes': list(range(2, 10)),
    'min_samples_split': [2, 3, 4],
    'max_depth': [5, 7, 9, 11]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=3
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42),
             param_grid={'max_depth': [5, 7, 9, 11],
                         'max_leaf_nodes': [2, 3, 4, 5, 6, 7, 8, 9],
                         'min_samples_split': [2, 3, 4]})

In [44]:
y_pred = grid_search.predict(X_test)

print("Roc auc:", round(roc_auc_score(y_test, y_pred), 2))
print("Best params:", grid_search.best_params_)

Roc auc: 0.7
Best params: {'max_depth': 5, 'max_leaf_nodes': 9, 'min_samples_split': 2}


In [46]:
rfc = RandomForestClassifier(random_state=31)
rfc.fit(X_train, y_train)

y_pred = rfc.predict(X_test)

print("Roc auc:", round(roc_auc_score(y_test, y_pred), 2))

Roc auc: 0.73


In [47]:
params = {
    'max_features': [ 4, 5, 6, 7],
    'min_samples_leaf': [3, 5, 7, 9, 11],
    'max_depth': [5, 10, 15]
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=31),
    param_grid=params,
    cv=3
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=31),
             param_grid={'max_depth': [5, 10, 15], 'max_features': [4, 5, 6, 7],
                         'min_samples_leaf': [3, 5, 7, 9, 11]})

In [48]:
y_pred = grid_search.predict(X_test)

print("Roc auc:", round(roc_auc_score(y_test, y_pred), 2))

Roc auc: 0.7


In [49]:
best_rfc = grid_search.best_estimator_
importances = best_rfc.feature_importances_
feature_names = X_train.columns

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
})
feat_imp = feat_imp.sort_values('importance', ascending=False)
print(feat_imp)

                 feature  importance
7            Humidity3pm    0.247119
2               Rainfall    0.080664
6            Humidity9am    0.074188
9            Pressure3pm    0.065947
10              Cloud9am    0.065545
..                   ...         ...
40    Location_GoldCoast    0.000000
69      Location_Walpole    0.000000
43   Location_Launceston    0.000000
72  Location_Witchcliffe    0.000000
52    Location_NorahHead    0.000000

[123 rows x 2 columns]
